# Imports

In [75]:
from pathlib import Path
import sys

root_dir = Path.cwd()

while not (root_dir / "setup.py").exists():
    root_dir = root_dir.parent

sys.path.append(str(root_dir / "src"))

print(root_dir)

/home/leonam/projects/insiders_clustering


In [76]:
# Path to the 'src' directory
SRC_DIR = root_dir / "src"

# Add 'src' to Python path so we can import modules like 'data.load_data'
sys.path.append(str(SRC_DIR))


In [77]:
import numpy                 as np
import pandas                as pd
import warnings
import pickle
import os
import s3fs

from dotenv                     import load_dotenv
from sqlalchemy                 import create_engine
from data.load_data             import load_ecommerce_data
from features.transform_columns import to_snake_case_columns
from sklearn.metrics            import silhouette_score


warnings.filterwarnings('ignore')


In [78]:
# Initialize the S3 filesystem client
fs = s3fs.S3FileSystem()

# S3 bucket used to store model artifacts
BUCKET = "leonam-insiders-dataset"

# Current model version used for artifact versioning
MODEL_VERSION = 1

# Load Dataset

In [108]:
path_s3 = "s3://leonam-insiders-dataset"

# Load data from Amazon S3
df_raw = load_ecommerce_data(f"{path_s3}/Ecommerce.csv")

df_raw.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,29-Nov-16,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,29-Nov-16,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,29-Nov-16,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,29-Nov-16,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,29-Nov-16,3.39,17850.0,United Kingdom


# <font color='blue'> 📊  1.0 Data Description

## <span style="color:blue">1.1</span> Rename Columns

In [80]:
df1 = df_raw.copy()

In [81]:
df1 = to_snake_case_columns(df1)
df1.head()

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,29-Nov-16,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,29-Nov-16,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,29-Nov-16,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,29-Nov-16,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,29-Nov-16,3.39,17850.0,United Kingdom


## <span style="color:blue">1.2</span> Data Dimensions

In [82]:
print(f'Number of rows: {df1.shape[0]}')
print(f'Number of columns: {df1.shape[1]}')

Number of rows: 541909
Number of columns: 8


## <span style="color:blue">1.3</span> Data Types

In [83]:
df1.dtypes

invoice_no       object
stock_code       object
description      object
quantity          int64
invoice_date     object
unit_price      float64
customer_id     float64
country          object
dtype: object

## <span style="color:blue">1.4</span> Check NA

In [84]:
df1.isna().sum()

invoice_no           0
stock_code           0
description       1454
quantity             0
invoice_date         0
unit_price           0
customer_id     135080
country              0
dtype: int64

## <span style="color:blue">1.5</span> Replace NA

In [85]:
# Get NA records
df_missing = df1.loc[df1['customer_id'].isna(), :]

# Get NOT NA records
df_not_missing  = df1.loc[~df1['customer_id'].isna(), :]

In [86]:
# create auxiliary dataframe with missing customer_id
df_backup = pd.DataFrame(df_missing['invoice_no'].drop_duplicates())
df_backup['customer_id'] = np.arange(19000, 19000 + len(df_backup), 1)

# merge auxiliary dataframe with df_missing
df_missing.drop(columns=['customer_id'], inplace=True)
df_missing = df_missing.merge(df_backup, on='invoice_no', how='left')

# concatenate df_not_missing and df_missing
df1 = pd.concat([df_not_missing, df_missing], axis=0)

df1.head()

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,29-Nov-16,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,29-Nov-16,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,29-Nov-16,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,29-Nov-16,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,29-Nov-16,3.39,17850.0,United Kingdom


In [87]:
df1.isna().sum()

invoice_no         0
stock_code         0
description     1454
quantity           0
invoice_date       0
unit_price         0
customer_id        0
country            0
dtype: int64

## <span style="color:blue">1.6</span> Change dtype

In [88]:
# invoice_date
df1['invoice_date'] = pd.to_datetime(df1['invoice_date'], format='%d-%b-%y')

# customer id
df1['customer_id'] = df1['customer_id'].astype(int)

df1.head()

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2016-11-29,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2016-11-29,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2016-11-29,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2016-11-29,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2016-11-29,3.39,17850,United Kingdom


In [89]:
df1.dtypes

invoice_no              object
stock_code              object
description             object
quantity                 int64
invoice_date    datetime64[ns]
unit_price             float64
customer_id              int64
country                 object
dtype: object

## <span style="color:blue">1.7</span> Descriptive Statistics

In [90]:
num_attributes = df1.select_dtypes(include=['int64', 'float64'])
cat_attributes = df1.select_dtypes(exclude=['int64', 'float64', 'datetime64[ns]'])

### <span style="color:blue">1.7.1</span> Numerical Attributes

In [91]:
# Calculate all desired statistics at once (central and dispersion)
stats = pd.DataFrame({
    'mean': num_attributes.mean(),
    'median': num_attributes.median(),
    'std': num_attributes.std(),
    'min': num_attributes.min(),
    'max': num_attributes.max(),
    'range': num_attributes.max() - num_attributes.min(),
    'skew': num_attributes.skew(),
    'kurtosis': num_attributes.kurtosis()
})

# Display the statistics
print("Numerical Attributes Statistics:")
stats

Numerical Attributes Statistics:


,mean,median,std,min,max,range,skew,kurtosis
quantity,9.552250,3.00,218.081158,-80995.00,80995.0,161990.00,-0.264076,119769.160031
unit_price,4.611114,2.08,96.759853,-11062.06,38970.0,50032.06,186.506972,59005.719097
customer_id,16688.840453,16249.00,2911.411352,12346.00,22709.0,10363.00,0.487449,-0.804287


### <span style="color:blue">1.7.2</span> Categorical Attributes

In [92]:
cat_attributes.head()

,invoice_no,stock_code,description,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,United Kingdom
1,536365,71053,WHITE METAL LANTERN,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,United Kingdom


In this first cycle, the Country column will not be used.

# <font color='blue'> 🧹2.0 Data Filtering

In [93]:
df2 = df1.copy()

In [94]:
df2.dtypes

invoice_no              object
stock_code              object
description             object
quantity                 int64
invoice_date    datetime64[ns]
unit_price             float64
customer_id              int64
country                 object
dtype: object

In [95]:

# === Numerical attributes ===
df2 = df2.loc[df2['unit_price'] >= 0.04, :]

# === Categorical attributes ====
df2 = df2[~df2['stock_code'].isin( ['POST', 'D', 'DOT', 'M', 'S', 'AMAZONFEE', 'm', 'DCGSSBOY', 'DCGSSGIRL', 'PADS', 'B', 'CRUK'] ) ]

# Remove column description
df2.drop(columns=['description'], inplace=True)

# country
df2 = df2[~df2['country'].isin(['European Community', 'Unspecified'])]

# bad users
df2 = df2[~df2['customer_id'].isin( [16446] )]

# quantity
df2_returns = df2.loc[df2['quantity'] < 0, :]
df2_purchases = df2.loc[df2['quantity'] > 0, :]


# <font color='blue'> ⚙️ 3.0 Feature Engineering

In [96]:
df3= df2.copy()

In [97]:
df3.shape

(536139, 7)

## <span style="color:blue">3.1</span> Feature Creation

In [98]:
# data reference
df_ref = (
    df3.drop( ['invoice_no', 'stock_code', 'quantity', 'invoice_date', 'unit_price', 'country'], axis=1 )
    .drop_duplicates( ignore_index=True )
)

### <span style="color:blue">3.1.1 </span> Gross Revenue



In [99]:
# Gross Revenue - Quantity * Unit Price
df2_purchases.loc[:, 'gross_revenue'] = df2_purchases.loc[:, 'quantity'] * df2_purchases.loc[:, 'unit_price']

# Monetary
df_monetary = df2_purchases.loc[:, ['customer_id', 'gross_revenue']].groupby( 'customer_id' ).sum().reset_index()
df_ref = pd.merge( df_ref, df_monetary, on='customer_id', how='left' )
df_ref.isna().sum()

customer_id       0
gross_revenue    91
dtype: int64

### <span style="color:blue">3.1.2</span> Recency - Day from Last Purchase

In [100]:
# Recency - Last Day Purchase
df_recency = (
    df2_purchases[['customer_id', 'invoice_date']]
    .groupby('customer_id').max().reset_index()
    .rename(columns={'invoice_date': 'last_day_purchase'})
)

df_recency['recency_days'] = (df2_purchases['invoice_date'].max() - df_recency['last_day_purchase']).dt.days
df_ref = pd.merge(df_ref, df_recency[['customer_id', 'recency_days']], on='customer_id', how='left')
df_ref.isna().sum()

customer_id       0
gross_revenue    91
recency_days     91
dtype: int64

### <span style="color:blue">3.1.5</span> Quantity of products purchased 

In [101]:
# Numero de produtos
df_freq = df2_purchases[['customer_id', 'stock_code']].\
            groupby( 'customer_id' ).count().\
            reset_index().\
            rename( columns={'stock_code': 'qtde_products'} )

df_ref = pd.merge( df_ref, df_freq, on='customer_id', how='left' )
df_ref.isna().sum()

customer_id       0
gross_revenue    91
recency_days     91
qtde_products    91
dtype: int64

### <span style="color:blue">3.1.8</span> Purchase Frequency

In [102]:
# Step 1: Create an auxiliary DataFrame with purchase activity summary for each customer
df_aux = (
    df2_purchases[['customer_id', 'invoice_no', 'invoice_date']]  # Select relevant columns
    .drop_duplicates()  # Remove duplicate purchase records
    .groupby('customer_id')  # Group by customer
    .agg(
        max_=('invoice_date', 'max'),  # Last purchase date
        min_=('invoice_date', 'min'),  # First purchase date
        days_=('invoice_date', lambda x: (x.max() - x.min()).days + 1),  # Total active days
        buy_=('invoice_no', 'count')  # Total number of purchases
    )
    .reset_index()
)

# Step 2: Calculate frequency of purchases (buys per day)
df_aux['frequency'] = df_aux.apply(
    lambda row: row['buy_'] / row['days_'] if row['days_'] != 0 else 0,
    axis=1
)

# Step 3: Merge frequency information into the reference DataFrame
df_ref = pd.merge(
    df_ref,
    df_aux[['customer_id', 'frequency']],
    on='customer_id',
    how='left'  # Keep all rows from df_ref
)

# Step 4: Check for missing values after the merge
df_ref.isna().sum()

customer_id       0
gross_revenue    91
recency_days     91
qtde_products    91
frequency        91
dtype: int64

### <span style="color:blue">3.1.9</span> Number of Returns

In [103]:
# Calculate total number of returned items per customer
df_returns = (
    df2_returns[['customer_id', 'quantity']]
    .groupby('customer_id')
    .sum()
    .reset_index()
    .rename(columns={'quantity': 'qtde_returns'})
)

# Convert return quantities to positive values
df_returns['qtde_returns'] = df_returns['qtde_returns'].abs()

# Merge return data into the reference DataFrame
df_ref = pd.merge(df_ref, df_returns, on='customer_id', how='left')

# Fill missing return quantities with 0 (i.e., no returns)
df_ref['qtde_returns'] = df_ref['qtde_returns'].fillna(0)

# Check for any remaining missing values in the DataFrame
df_ref.isna().sum()

customer_id       0
gross_revenue    91
recency_days     91
qtde_products    91
frequency        91
qtde_returns      0
dtype: int64

# <font color='blue'> 🎯 5.0 Feature Spotlight: Selecting the Best Predictors

In [104]:
df5 = df_ref.dropna().copy()

cols_selected = ['customer_id', 'gross_revenue', 'recency_days', 'qtde_products', 'frequency', 'qtde_returns'] 

df5 = df5[cols_selected].copy()
df5.head()

,customer_id,gross_revenue,recency_days,qtde_products,frequency,qtde_returns
0,17850,5391.21,372.0,297.0,17.000000,40.0
1,13047,3232.59,56.0,171.0,0.028302,35.0
2,12583,6705.38,2.0,232.0,0.040323,50.0
3,13748,948.25,95.0,28.0,0.017921,0.0
4,15100,876.00,333.0,3.0,0.073171,22.0


In [109]:
df5.shape

(5695, 6)

# <font color='blue'> 🛠️ 6.0 Data Transforming

In [110]:
# Select the columns to be used for clustering
cols_selected = ['customer_id', 'gross_revenue', 'recency_days', 'qtde_products', 'frequency', 'qtde_returns'] 

df6 = df5[cols_selected].copy()

In [111]:
# Load and apply gross_revenue scaler
# gross_revenue_scaler = pickle.load(open(SRC_DIR / 'features/gross_revenue_scaler.pkl', 'rb'))
with fs.open(f'{BUCKET}/models/v{MODEL_VERSION}/scalers/gross_revenue_scaler.pkl', 'rb') as f:
    gross_revenue_scaler = pickle.load(f)

df6['gross_revenue'] = gross_revenue_scaler.transform(df6[['gross_revenue']])

# Load and apply recency_days scaler
# recency_days_scaler = pickle.load(open(SRC_DIR / 'features/recency_days_scaler.pkl', 'rb'))
with fs.open(f'{BUCKET}/models/v{MODEL_VERSION}/scalers/recency_days_scaler.pkl', 'rb') as f:
    recency_days_scaler = pickle.load(f)

df6['recency_days'] = recency_days_scaler.transform(df6[['recency_days']])

# Load and apply qtde_products scaler
# qtde_products_scaler = pickle.load(open(SRC_DIR / 'features/qtde_products_scaler.pkl', 'rb'))
with fs.open(f'{BUCKET}/models/v{MODEL_VERSION}/scalers/qtde_products_scaler.pkl', 'rb') as f:
    qtde_products_scaler = pickle.load(f)

df6['qtde_products'] = qtde_products_scaler.transform(df6[['qtde_products']])

# Load and apply frequency scaler
# frequency_scaler = pickle.load(open(SRC_DIR / 'features/frequency_scaler.pkl', 'rb'))
with fs.open(f'{BUCKET}/models/v{MODEL_VERSION}/scalers/frequency_scaler.pkl', 'rb') as f:
    frequency_scaler = pickle.load(f)

df6['frequency'] = frequency_scaler.transform(df6[['frequency']])

# Load and apply qtde_returns scaler
# qtde_returns_scaler = pickle.load(open(SRC_DIR / 'features/qtde_returns_scaler.pkl', 'rb'))
with fs.open(f'{BUCKET}/models/v{MODEL_VERSION}/scalers/qtde_returns_scaler.pkl', 'rb') as f:
    qtde_returns_scaler = pickle.load(f)

df6['qtde_returns'] = qtde_returns_scaler.transform(df6[['qtde_returns']])

# Check the first few rows of the scaled DataFrame
df6.head()

,customer_id,gross_revenue,recency_days,qtde_products,frequency,qtde_returns
0,17850,0.019312,0.997319,0.037770,1.000000,0.000539
1,13047,0.011579,0.150134,0.021692,0.001345,0.000472
2,12583,0.024020,0.005362,0.029476,0.002052,0.000674
3,13748,0.003396,0.254692,0.003445,0.000734,0.000000
4,15100,0.003137,0.892761,0.000255,0.003985,0.000296


In [113]:
X = df6.drop(columns=['customer_id']).copy()

# Fit UMAP picle file
# reducer = pickle.load(open(SRC_DIR / 'models/umap_embedding.pkl', 'rb'))
with fs.open(f'{BUCKET}/models/v{MODEL_VERSION}/embeddings/umap_embedding.pkl', 'rb') as f:
    reducer = pickle.load(f)

embedding = reducer.transform(X)       # Transform the data into 2D using the pre-fitted UMAP model

# Create a new DataFrame with the UMAP embeddings
df_umap = pd.DataFrame({
    'embedding_x': embedding[:, 0],        # First UMAP component (x-axis)
    'embedding_y': embedding[:, 1]         # Second UMAP component (y-axis)
})


# <font color='blue'> 🚀 9.0 Model Training

In [115]:
X = df_umap.copy()

## <span style="color:blue">9.1</span>  Kmeans

In [116]:
# Define the number of clusters
k = 10

# artifact = pickle.load(open(SRC_DIR / 'models/kmeans_model.pkl', 'rb'))
with fs.open(f'{BUCKET}/models/v{MODEL_VERSION}/kmeans_model.pkl', 'rb') as f:
    kmeans_model = pickle.load(f)

# Fit the model
kmeans_model.fit(X)

# Get the cluster labels
labels = kmeans_model.labels_

# Check unique clusters
print(np.unique(labels))

[0 1 2 3 4 5 6 7 8 9]


## <span style="color:blue">9.2</span> Cluster Validation

In [117]:
## WSS (Within-cluster sum of squares)
# print('WSS value: ', model_kmeans.inertia_)

## Silhouette Score
ss = silhouette_score(X, labels, metric='euclidean')
print(f"Silhouette Score (SS): {ss:.3f}")

Silhouette Score (SS): 0.481


# <font color='blue'> 🌐 10.0 Cluster Analysis: Unveiling Hidden Patterns

In [118]:
df10 = df5.copy()
df10['cluster'] = labels
df10.head()

,customer_id,gross_revenue,recency_days,qtde_products,frequency,qtde_returns,cluster
0,17850,5391.21,372.0,297.0,17.000000,40.0,0
1,13047,3232.59,56.0,171.0,0.028302,35.0,4
2,12583,6705.38,2.0,232.0,0.040323,50.0,1
3,13748,948.25,95.0,28.0,0.017921,0.0,4
4,15100,876.00,333.0,3.0,0.073171,22.0,2


## <span style="color:blue">10.2</span> Cluster Profile

In [119]:
# Number os customers
df_cluster = df10[['cluster', 'customer_id']].groupby('cluster').count().reset_index().rename(columns={'customer_id': 'count_customer'})
df_cluster['perc_customer'] = df_cluster['count_customer'] / df_cluster['count_customer'].sum() * 100

# Avg Gross Revenue
df_cluster['avg_gross_revenue'] = df10[['cluster', 'gross_revenue']].groupby('cluster').mean().reset_index()['gross_revenue']

# Avg Recency Days
df_cluster['avg_recency_days'] = df10[['cluster', 'recency_days']].groupby('cluster').mean().reset_index()['recency_days']

# Avg Quantity of Products
df_cluster['avg_qtde_products'] = df10[['cluster', 'qtde_products']].groupby('cluster').mean().reset_index()['qtde_products']

# Avg Frequency
df_cluster['avg_frequency'] = df10[['cluster', 'frequency']].groupby('cluster').mean().reset_index()['frequency']

# Avg Returns
df_cluster['avg_returns'] = df10[['cluster', 'qtde_returns']].groupby('cluster').mean().reset_index()['qtde_returns']


# Store old cluster labels
df_cluster['old_cluster'] = df_cluster['cluster']

# Sort clusters by avg_gross_revenue (descending) and reassign cluster labels
df_cluster = (
    df_cluster
    .sort_values('avg_gross_revenue', ascending=False)
    .reset_index(drop=True)
)

# Assign cluster labels starting from 1
df_cluster['cluster'] = df_cluster.index + 1

# Create mapping dictionary
cluster_map = dict(zip(df_cluster['old_cluster'], df_cluster['cluster']))

# Apply mapping safely
df10['cluster_new'] = df10['cluster'].map(cluster_map)

# Check for unmapped clusters
print("Unmapped clusters:")
print(df10[df10['cluster_new'].isna()]['cluster'].unique())

# Replace old cluster
df10['cluster'] = df10['cluster_new']
df10.drop(columns=['cluster_new'], inplace=True)

# Drop helper column
df_cluster.drop(columns=['old_cluster'], inplace=True)

df_cluster


Unmapped clusters:
[]


,cluster,count_customer,perc_customer,avg_gross_revenue,avg_recency_days,avg_qtde_products,avg_frequency,avg_returns
0,1,825,14.486392,6104.127976,6.122424,236.220606,0.049990,74.030303
1,2,922,16.189640,1809.660683,27.048807,107.749458,0.032102,16.373102
2,3,508,8.920105,1452.534193,68.836614,75.208661,0.025946,10.025591
3,4,426,7.480246,1023.413310,59.553991,97.176056,1.010507,6.734742
4,5,340,5.970149,1020.148265,337.341176,61.388235,0.862909,30.141176
5,6,366,6.426690,981.871093,147.139344,56.021858,0.037320,9.959016
6,7,378,6.637401,899.613333,310.753968,61.854497,0.833050,202.955026
7,8,761,13.362599,690.730749,146.972405,57.069645,1.018397,1.630749
8,9,498,8.744513,601.480602,24.024096,34.616466,1.049029,1.248996
9,10,671,11.782265,528.959270,247.019374,41.959762,1.017884,2.186289


### <span style="color:blue">10.2.1</span> Customer Segmentation Results 


#### 📊 Cluster Profiling (10 Clusters)

Below is the corrected and aligned interpretation of each cluster based on the table provided.

---

##### 🥇 **Cluster 1 – Premium High-Value Customers**
- **Represent:** 14.5% of customers
- **Highest average revenue:** ≈ 6,104
- **Very recent activity:** ≈ 6 days
- **Very high purchase volume:** ≈ 236 products
- **High returns:** ≈ 74

**Profile:**  
The most valuable customer segment, combining the highest revenue, very recent purchases, and strong purchasing activity. Although returns are relatively high, they are proportional to their purchasing volume.

**➡ Top Priority Segment**

---

##### 🥈 **Cluster 2 – Active Mid-Value Customers**
- **Represent:** 16.2% of customers
- **Revenue:** ≈ 1,810
- **Recent activity:** ≈ 27 days
- **High purchase volume:** ≈ 108 products
- **Moderate returns:** ≈ 16

**Profile:**  
Active customers with solid spending and consistent purchasing behavior. A large and valuable segment to retain through loyalty initiatives.

**➡ Retain and Increase Lifetime Value**

---

##### 🥉 **Cluster 3 – Moderate-Value Regular Buyers**
- **Represent:** 8.9% of customers
- **Revenue:** ≈ 1,453
- **Recency:** ≈ 69 days
- **Moderate purchase volume:** ≈ 75 products
- **Low returns:** ≈ 10

**Profile:**  
Customers with moderate revenue and engagement but showing signs of reduced activity.

**➡ Reactivation Opportunity**

---

##### 🔹 **Cluster 4 – Frequent Low-Mid Value Buyers**
- **Represent:** 7.5% of customers
- **Revenue:** ≈ 1,023
- **Recency:** ≈ 60 days
- **High purchase volume:** ≈ 97 products
- **Low returns:** ≈ 7

**Profile:**  
Customers with frequent purchases and relatively low operational cost, despite generating moderate revenue.

**➡ Retain Through Engagement**

---

##### 🔹 **Cluster 5 – Dormant High-Return Customers**
- **Represent:** 6.0% of customers
- **Revenue:** ≈ 1,020
- **Very long recency:** ≈ 337 days
- **Moderate purchase volume:** ≈ 61 products
- **High returns:** ≈ 30

**Profile:**  
Previously active customers who have become inactive and exhibit relatively high return rates.

**➡ Churn Risk – Reactivation Campaigns**

---

##### 🔹 **Cluster 6 – Inactive Moderate Buyers**
- **Represent:** 6.4% of customers
- **Revenue:** ≈ 982
- **Long recency:** ≈ 147 days
- **Moderate purchase volume:** ≈ 56 products
- **Low returns:** ≈ 10

**Profile:**  
Customers with moderate historical value but low recent engagement.

**➡ Win-Back Opportunity**

---

##### 🔹 **Cluster 7 – High-Return Heavy Buyers**
- **Represent:** 6.6% of customers
- **Revenue:** ≈ 900
- **Very long recency:** ≈ 311 days
- **Moderate purchase volume:** ≈ 62 products
- **Extremely high returns:** ≈ 203

**Profile:**  
Customers with exceptionally high return rates and low recent activity, making them operationally expensive.

**➡ High Operational Risk**

---

##### 🔹 **Cluster 8 – Low-Value Frequent Buyers**
- **Represent:** 13.4% of customers
- **Revenue:** ≈ 691
- **Long recency:** ≈ 147 days
- **Moderate purchase volume:** ≈ 57 products
- **Very low returns:** ≈ 2

**Profile:**  
Large customer segment with low spending but low operational cost.

**➡ Maintain with Low-Cost Campaigns**

---

##### ⚠️ **Cluster 9 – Low-Value Recent Customers**
- **Represent:** 8.7% of customers
- **Revenue:** ≈ 601
- **Recent activity:** ≈ 24 days
- **Low purchase volume:** ≈ 35 products
- **Minimal returns:** ≈ 1

**Profile:**  
Low-spending customers who remain active and may have potential for future growth.

**➡ Upselling Opportunity**

---

##### ❌ **Cluster 10 – Low-Value Infrequent Buyers**
- **Represent:** 11.8% of customers
- **Revenue:** ≈ 529
- **Long recency:** ≈ 247 days
- **Low purchase volume:** ≈ 42 products
- **Very low returns:** ≈ 2

**Profile:**  
Low-value customers with infrequent purchases and limited recent engagement.

**➡ Lowest Priority Segment**

---

# <font color='blue'> ✨ 11.0 Deploy to Production

## 11.1 Insert Into SQLITE

In [120]:
# change dtypes
df10['recency_days'] = df10['recency_days'].astype(int)
df10['qtde_products'] = df10['qtde_products'].astype(int)
df10['qtde_returns'] = df10['qtde_returns'].astype(int)

In [121]:
df10.head()

,customer_id,gross_revenue,recency_days,qtde_products,frequency,qtde_returns,cluster
0,17850,5391.21,372,297,17.000000,40,5
1,13047,3232.59,56,171,0.028302,35,3
2,12583,6705.38,2,232,0.040323,50,1
3,13748,948.25,95,28,0.017921,0,3
4,15100,876.00,333,3,0.073171,22,7


In [122]:
# ==============================================================================
# DATABASE CONNECTION SETUP
# Uncomment the block for the database you wish to use and comment out the other.
# ==============================================================================

# --- OPTION 1: AWS RDS PostgreSQL ---
load_dotenv()
HOST = os.getenv("DB_HOST")
PORT = os.getenv("DB_PORT", "5432")
DATABASE = os.getenv("DB_NAME")
USER = os.getenv("DB_USER")
PASSWORD = os.getenv("DB_PASSWORD")

db_url = f"postgresql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
engine = create_engine(db_url)


# --- OPTION 2: Local SQLite ---
# sqlite_path = root_dir / "data" / "processed" / "insiders_db.sqlite"
# db_url = f"sqlite:///{sqlite_path}"
# engine = create_engine(db_url)

In [ ]:
# ==============================================================================
# DROP TABLE
# ==============================================================================

# with engine.begin() as conn:
#     conn.exec_driver_sql("DROP TABLE IF EXISTS insiders;")

In [124]:
# ==============================================================================
# CREATE TABLE IF NOT EXISTS
# ==============================================================================

create_table_sql = """
CREATE TABLE IF NOT EXISTS insiders (
    customer_id BIGINT NOT NULL,
    gross_revenue DOUBLE PRECISION,
    recency_days INTEGER,
    qtde_products INTEGER,
    frequency DOUBLE PRECISION,
    qtde_returns INTEGER,
    cluster INTEGER,
    model_version INTEGER NOT NULL,
    prediction_date TIMESTAMP NOT NULL,
    PRIMARY KEY (customer_id, model_version)
);
"""

with engine.begin() as conn:
    conn.exec_driver_sql(create_table_sql)

In [125]:
# ==============================================================================
# DATA INGESTION
# ==============================================================================

# Add metadata columns
df10["model_version"] = MODEL_VERSION
df10["prediction_date"] = pd.Timestamp.now()

# Load existing primary keys from output table
query = """
SELECT customer_id, model_version
FROM insiders;
"""

df_existing = pd.read_sql(query, con=engine)

# Left join to identify new records only
df_to_insert = df10.merge(
    df_existing,
    on=["customer_id", "model_version"],
    how="left",
    indicator=True
)

# Keep only records that do not exist in the database
df_to_insert = (
    df_to_insert[df_to_insert["_merge"] == "left_only"]
    .drop(columns=["_merge"])
)

# Append only new records
if not df_to_insert.empty:
    df_to_insert.to_sql(
        name="insiders",
        con=engine,
        if_exists="append",
        index=False,
        method="multi"
    )

In [126]:
df_to_insert.head()

,customer_id,gross_revenue,recency_days,qtde_products,frequency,qtde_returns,cluster,model_version,prediction_date


In [73]:
# ==============================================================================
# VERIFICATION & QUERIES
# ==============================================================================

# Fetch and display the first 5 rows to verify insertion
df_check = pd.read_sql_query("SELECT * FROM insiders LIMIT 5;", con=engine)
print("Preview of inserted data:")
print(df_check)

# Check the total row count in the table
total_rows = pd.read_sql_query("SELECT COUNT(*) FROM insiders;", con=engine)
print("\nTotal row count:")
print(total_rows)


# ==============================================================================
# CLEANUP
# ==============================================================================

# Dispose of the connection pool to free up system and database resources
engine.dispose()

Preview of inserted data:
   customer_id  gross_revenue  recency_days  qtde_products  frequency  \
0        17850        5391.21           372            297  17.000000   
1        13047        3232.59            56            171   0.028302   
2        12583        6705.38             2            232   0.040323   
3        13748         948.25            95             28   0.017921   
4        15100         876.00           333              3   0.073171   

   qtde_returns  cluster  model_version            prediction_date  
0            40        5              1 2026-07-27 20:56:28.939400  
1            35        3              1 2026-07-27 20:56:28.939400  
2            50        1              1 2026-07-27 20:56:28.939400  
3             0        3              1 2026-07-27 20:56:28.939400  
4            22        7              1 2026-07-27 20:56:28.939400  

Total row count:
   count
0   5695
